# 01 - Qwen3 Architecture Overview

**Goal:** Get hands-on with the Qwen3 model family. Understand the architecture by inspecting real model configs and weights, not just reading about them.

**What we'll do:**
1. Survey the Qwen3 model family (dense vs MoE, sizes)
2. Load and inspect a model config (zero-cost, no weights)
3. Load the smallest model (Qwen3-0.6B) and walk through its layers
4. Run a simple inference to see it in action
5. Explore thinking mode (`/think` vs `/no_think`)

**Prerequisites:** Basic understanding of Transformers and attention mechanisms.

## Setup

In [1]:
import sys
sys.path.insert(0, "..")  # Allow importing from src/

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from src.utils import get_device, print_model_summary, format_param_count

device = get_device()
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

/Users/treese/Library/Mobile Documents/com~apple~CloudDocs/treese/Projects (Cloud)/Study Grounds/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps
PyTorch version: 2.10.0


## The Qwen3 Model Family

Qwen3 comes in two flavors: **Dense** models (every parameter active on every token) and **MoE** models (Mixture of Experts — only a subset of parameters activate per token).

| Model | Type | Total Params | Active Params | Local Feasibility |
|-------|------|-------------|---------------|-------------------|
| Qwen3-0.6B | Dense | 0.6B | 0.6B | Easy |
| Qwen3-1.7B | Dense | 1.7B | 1.7B | Easy |
| Qwen3-4B | Dense | 4B | 4B | OK (16GB+ RAM) |
| Qwen3-8B | Dense | 8B | 8B | Tight (16GB), OK (32GB+) |
| Qwen3-14B | Dense | 14B | 14B | Config-only locally |
| Qwen3-32B | Dense | 32B | 32B | Config-only locally |
| Qwen3-30B-A3B | MoE | 30B | 3B | Feasible (float16) |
| Qwen3-235B-A22B | MoE | 235B | 22B | Config-only locally |

We'll start with **Qwen3-0.6B** for hands-on inspection, and look at configs of larger models to understand architectural differences.

## Part 1: Inspecting the Architecture (Config Only)

Loading just the config downloads a tiny JSON file — no model weights. This lets us examine the architecture of ANY model, regardless of size.

In [2]:
# Load the config for the smallest dense model
config_06b = AutoConfig.from_pretrained("Qwen/Qwen3-0.6B")
print(config_06b)

Qwen3Config {
  "architectures": [
    "Qwen3ForCausalLM"
  ],
  "attention_bias": false,
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "head_dim": 128,
  "hidden_act": "silu",
  "hidden_size": 1024,
  "initializer_range": 0.02,
  "intermediate_size": 3072,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention"
  ],
  "max_position_embeddings": 40960,
  "max_w

### Key Config Fields Explained

| Field | What It Means |
|-------|---------------|
| `hidden_size` | Dimension of the residual stream (d_model) |
| `num_hidden_layers` | Number of transformer blocks stacked |
| `num_attention_heads` | Number of query heads in multi-head attention |
| `num_key_value_heads` | Number of KV heads (< num_attention_heads = Grouped Query Attention) |
| `intermediate_size` | Hidden dim of the feed-forward network (usually ~4x hidden_size) |
| `vocab_size` | Size of the token vocabulary |
| `max_position_embeddings` | Maximum sequence length the model supports |
| `rope_theta` | Base frequency for Rotary Position Embeddings |

Let's pull these out cleanly:

In [ ]:
def summarize_config(config, name="Model"):
    """Print a clean summary of a model config."""
    print(f"=== {name} ===")
    print(f"  Hidden size:          {config.hidden_size}")
    print(f"  Num layers:           {config.num_hidden_layers}")
    print(f"  Attention heads (Q):  {config.num_attention_heads}")
    print(f"  KV heads:             {config.num_key_value_heads}")
    print(f"  GQA ratio:            {config.num_attention_heads // config.num_key_value_heads}:1")
    print(f"  FFN intermediate:     {config.intermediate_size}")
    print(f"  Vocab size:           {config.vocab_size:,}")
    print(f"  Max seq length:       {config.max_position_embeddings:,}")
    print(f"  RoPE theta:           {config.rope_theta:,.0f}")
    print()

summarize_config(config_06b, "Qwen3-0.6B")

### Comparing Dense vs MoE Configs

Let's compare a dense model with an MoE model to see the architectural differences:

In [ ]:
# Load MoE config for comparison
config_moe = AutoConfig.from_pretrained("Qwen/Qwen3-30B-A3B")

summarize_config(config_06b, "Qwen3-0.6B (Dense)")
summarize_config(config_moe, "Qwen3-30B-A3B (MoE)")

# MoE-specific fields
if hasattr(config_moe, "num_experts"):
    print(f"--- MoE-Specific Fields ---")
    print(f"  Num experts:          {config_moe.num_experts}")
if hasattr(config_moe, "num_experts_per_tok"):
    print(f"  Experts per token:    {config_moe.num_experts_per_tok}")
if hasattr(config_moe, "decoder_sparse_step"):
    print(f"  Sparse step:          {config_moe.decoder_sparse_step}")

## Part 2: Loading and Inspecting Qwen3-0.6B

Now let's load the actual model weights. Qwen3-0.6B is ~1.2GB in float16 — should load quickly.

In [ ]:
model_name = "Qwen/Qwen3-0.6B"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)

print("\nModel loaded!")
print_model_summary(model)

### Layer-by-Layer Inspection

Let's walk through every layer in the model to understand the architecture:

In [ ]:
# Print the high-level module structure
for name, module in model.named_children():
    print(f"{name}: {type(module).__name__}")
    for child_name, child in module.named_children():
        param_count = sum(p.numel() for p in child.parameters())
        print(f"  {child_name}: {type(child).__name__} ({format_param_count(param_count)})")

In [ ]:
# Detailed look at a single transformer block
block = model.model.layers[0]
print(f"=== Transformer Block 0 ===")
print(block)
print()

# Parameter breakdown for this block
for name, param in block.named_parameters():
    print(f"  {name}: {list(param.shape)} ({format_param_count(param.numel())})")

### Observations

*After running the cells above, note your observations here:*

- What attention mechanism does it use? (MHA, GQA, MQA?)
- What is the FFN structure? (Standard, SwiGLU, etc.)
- What normalization is used? (LayerNorm, RMSNorm?)
- How are position embeddings handled? (Absolute, RoPE, ALiBi?)

## Part 3: Simple Inference

Let's see the model generate text to confirm everything works:

In [ ]:
prompt = "The Transformer architecture is important because"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        do_sample=True,
        temperature=0.7,
        top_p=0.9,
    )

response = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(response)

## Part 4: Thinking Mode

Qwen3 models support a unified thinking/non-thinking mode. The model can be prompted with special tokens to enable or disable its internal chain-of-thought reasoning.

- `/think` — enables extended reasoning (the model "thinks" before answering)
- `/no_think` — direct response without explicit reasoning

Let's see if this works with the 0.6B model (note: smaller models may not exhibit strong thinking behavior):

In [ ]:
# Check if the tokenizer has the thinking tokens
special_tokens = tokenizer.all_special_tokens
thinking_tokens = [t for t in special_tokens if "think" in t.lower()]
print(f"Thinking-related special tokens: {thinking_tokens}")
print(f"\nAll special tokens: {special_tokens}")

In [ ]:
# Try a chat-format prompt with thinking enabled
messages = [
    {"role": "user", "content": "What is 15 * 37? /think"}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)
print("Formatted prompt:")
print(repr(text))
print()

inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=True,
        temperature=0.7,
    )

response = tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=False)
print("Response (with special tokens visible):")
print(response)

## Part 5: Tokenizer Deep Dive

Understanding the tokenizer is important for analyzing how the model processes input:

In [ ]:
print(f"Vocab size: {tokenizer.vocab_size:,}")
print(f"Model max length: {tokenizer.model_max_length:,}")
print(f"Tokenizer class: {type(tokenizer).__name__}")
print()

# Tokenize a sample and see the tokens
sample = "Mixture of Experts allows models to scale parameters without scaling compute."
tokens = tokenizer.tokenize(sample)
token_ids = tokenizer.encode(sample)

print(f"Text: {sample}")
print(f"Tokens ({len(tokens)}): {tokens}")
print(f"Token IDs: {token_ids}")

---

## What I Learned

*Fill this in after running through the notebook:*

- 
- 
- 

## Questions for Next Time

- How does the MoE routing actually work at inference time?
- What do the attention patterns look like? Are some heads specialized?
- How does the model's architecture compare to Llama 3, Gemma 2, etc.?

## Next Steps

- **02-moe-exploration.ipynb** — Deep dive into the MoE architecture of Qwen3-30B-A3B
- **03-attention-analysis.ipynb** — Visualize attention patterns and head specialization